In [ ]:
%pip install --upgrade transformers datasets tensorflow

In [ ]:
token_value = None

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# Load the tokenizer

tokenizer = AutoTokenizer.from_pretrained("epfl-llm/meditron-7b", token= token_value)

In [ ]:
# Load the model

model = AutoModelForCausalLM.from_pretrained("epfl-llm/meditron-7b", token= token_value)

In [ ]:
#!git clone https://github.com/*****/meditron.git

In [ ]:
#%cd 'meditron'

In [ ]:
!git clone https://github.com/****/FastChat.git

In [ ]:
!chmod -R +x 'FastChat'

In [ ]:
%cd 'FastChat'

In [ ]:
HF_TOKEN = None

In [ ]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [ ]:
%pip install --upgrade httpcore fastapi uvicorn pyngrok shortuuid websockets

In [ ]:
!python3 -m pip install -e ".[model_worker,webui]" --quiet

In [ ]:
!pwd

In [ ]:
import subprocess
import threading

# Using 127.0.0.1 because localhost does not work properly in Colab

def run_controller():
    subprocess.run(["python3", "-m", "fastchat.serve.controller", "--host", "127.0.0.1"])

def run_model_worker():
    subprocess.run(["python3", "-m", "fastchat.serve.model_worker", "--host", "127.0.0.1", "--controller-address", "http://127.0.0.1:21001", "--model-path", "epfl-llm/meditron-7b", "--device", "cpu", "--load-8bit"])

def run_api_server():
    subprocess.run(["python3", "-m", "fastchat.serve.openai_api_server", "--host", "127.0.0.1", "--controller-address", "http://127.0.0.1:21001", "--port", "8000"])


In [ ]:
# Start controller
# see `controller.log`

controller_thread = threading.Thread(target=run_controller)
controller_thread.start()

In [ ]:
# Start model worker
# see `controller.log`

# IMPORTANT: to wait until the checkpoint shards are fully downloaded!!
model_worker_thread = threading.Thread(target=run_model_worker)
model_worker_thread.start()

In [ ]:
%pip install pydantic_settings

In [ ]:
%pip install -U email-validator

In [ ]:
# Start API server

api_server_thread = threading.Thread(target=run_api_server)
api_server_thread.start()

In [ ]:
!curl http://127.0.0.1:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{ \
    "model": "meditron-7b", \
    "messages": [{"role": "user", "content": "What is the treatment for cold?"}], \
    "temperature": 0.5 \
  }'

In [ ]:
!curl http://127.0.0.1:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{ \
    "model": "meditron-7b", \
    "messages": [{"role": "user", "content": "Hello, How are you?"}], \
    "temperature": 0.5 \
  }'

In [ ]:
%pip install deep_translator

In [ ]:
from deep_translator import GoogleTranslator

translator_en_to_fa = GoogleTranslator(source='auto', target='fa')
translator_fa_to_en = GoogleTranslator(source='fa', target='en')

def translate_text(text, des, chunk_size=4999):
    if isinstance(text, str):
        chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
        if des == 'fa':
            translated_chunks = [translator_en_to_fa.translate(chunk) for chunk in chunks]
        elif des == 'en':
            translated_chunks = [translator_fa_to_en.translate(chunk) for chunk in chunks]
        return ''.join(translated_chunks)

In [ ]:
import requests
import json

def make_chat_completion_request(url, model, message, temperature=0.5):
    headers = {
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": [{"role": "user", "content": message}],
        "temperature": temperature
    }
    try:
        # Send POST request to the API endpoint
        response = requests.post(url, headers=headers, data=json.dumps(data))

        if response.status_code == 200:
            return response.json()  # Return the response data in JSON format
        else:
            Exception(f"Error: {response.status_code}, {response.text}")
    except Exception as e:
        Exception(f"An error occurred: {e}")

In [ ]:
url = "http://127.0.0.1:8000/v1/chat/completions"
model = "meditron-7b"
message = "Hello, can you tell me a joke for me?"
temperature = 0.5

response_data = make_chat_completion_request(url, model, message, temperature)

In [ ]:
print( response_data['choices'][0]['message']['content'] )

In [ ]:
url = "http://127.0.0.1:8000/v1/chat/completions"
model = "meditron-7b"
message = "Hello, How are you?"
temperature = 0.5

response_data = make_chat_completion_request(url, model, message, temperature)

In [ ]:
print(response_data['choices'][0]['message']['content'])

In [ ]:
def persian_meditron(question, temperature = 0.5):
    url = "HF_TOKEN"
    model = "meditron-7b"
    question = translate_text(question, 'en')
    response_data = make_chat_completion_request(url, model, question, temperature)
    answer = response_data['choices'][0]['message']['content']
    try:
        return translate_text(answer, des = 'fa')
    except:
        Exception('Sorry! Can Not be done At this time!')

In [ ]:
from google.colab import files
from PIL import Image
import torchvision.transforms as transforms
import torch
import torchvision.models as models
import datetime


uploaded = files.upload()

file_name = list(uploaded.keys())[0]
image = Image.open(file_name)


image.show()


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

image_tensor = transform(image).unsqueeze(0)

model = models.resnet18(pretrained=True)
model.eval()


with torch.no_grad():
    outputs = model(image_tensor)
    predicted_class = torch.argmax(outputs, 1).item()


disease_classes = [
    'سرطان پوست', 'دیابت چشمی', 'آرتریت', 'بیماری قلبی',
    'عفونت ریوی', 'مشکلات کبدی', 'سکته مغزی', 'سرطان ریه'
]

# تشخیص اولیه
disease_name = disease_classes[predicted_class % len(disease_classes)]

# =============================
# تابع برای گرفتن توضیحات فارسی از مدیترون (اینجا باید به مدیترون متصل بشه)
# =============================

def get_disease_description_farsi(disease_name):
    # اینجا بجای return ثابت، میتونی با کدهای قبلی مدیترون، query بفرستی و توضیحات فارسی بگیری
    # فعلاً نمونه فیک:
    descriptions = {
        'سرطان پوست': 'سرطان پوست یکی از شایع‌ترین انواع سرطان است که عمدتاً به دلیل قرار گرفتن در معرض نور خورشید ایجاد می‌شود.',
        'دیابت چشمی': 'دیابت چشمی یک عارضه مرتبط با دیابت است که باعث آسیب به عروق خونی شبکیه چشم می‌شود.',
        'آرتریت': 'آرتریت التهاب مفاصل است که می‌تواند باعث درد و محدودیت حرکت شود.',
        'بیماری قلبی': 'بیماری قلبی شامل مشکلات و عارضه‌های قلب و عروق است که می‌تواند تهدیدکننده حیات باشد.',
        'عفونت ریوی': 'عفونت ریوی نوعی عفونت است که در ریه‌ها رخ می‌دهد و معمولاً به دلیل باکتری یا ویروس ایجاد می‌شود.',
        'مشکلات کبدی': 'بیماری‌های کبدی شامل عفونت، التهاب و نارسایی‌های عملکردی کبد است.',
        'سکته مغزی': 'سکته مغزی زمانی رخ می‌دهد که خون‌رسانی به بخشی از مغز مختل شده یا کاهش یابد.',
        'سرطان ریه': 'سرطان ریه یکی از شایع‌ترین سرطان‌ها است که معمولاً در افراد سیگاری دیده می‌شود.'
    }
    return descriptions.get(disease_name, 'اطلاعاتی موجود نیست')

disease_description_fa = get_disease_description_farsi(disease_name)

# =============================
# ایجاد گزارش نهایی
# =============================

now = datetime.datetime.now()
report = f"""
گزارش تشخیص پزشکی
----------------------
تاریخ و زمان: {now.strftime('%Y-%m-%d %H:%M:%S')}
نام بیماری: {disease_name}
توضیحات: {disease_description_fa}
----------------------
"""

# چاپ گزارش
print(report)

# ذخیره در فایل
with open('medical_report.txt', 'w', encoding='utf-8') as file:
    file.write(report)

print("✅ گزارش در فایل 'medical_report.txt' ذخیره شد.")


In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# Configure quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model and tokenizer
model = AutoModel.from_pretrained(
    "ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
)
tokenizer = AutoTokenizer.from_pretrained("ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1", trust_remote_code=True)

# Load an image and prepare a question
image = Image.open("Path to Your image").convert('RGB')
question = 'Give the modality, organ, analysis, abnormalities (if any), treatment (if abnormalities are present)?'

# Format the input message
msgs = [{'role': 'user', 'content': [image, question]}]

# Generate a response using the model
res = model.chat(
    image=image,
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True,
    temperature=0.95,
    stream=True
)

# Collect and print the generated response
generated_text = ""
for new_text in res:
    generated_text += new_text
    print(new_text, flush=True, end='')
print(generated_text)

In [ ]:
pip install bitsandbytes

In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# Configure quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model and tokenizer
model = AutoModel.from_pretrained(
    "ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
)
tokenizer = AutoTokenizer.from_pretrained("ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1", trust_remote_code=True)

# Load an image and prepare a question
image = Image.open("Path to Your image").convert('RGB')
question = 'Give the modality, organ, analysis, abnormalities (if any), treatment (if abnormalities are present)?'

# Format the input message
msgs = [{'role': 'user', 'content': [image, question]}]

# Generate a response using the model
res = model.chat(
    image=image,
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True,
    temperature=0.95,
    stream=True
)

# Collect and print the generated response
generated_text = ""
for new_text in res:
    generated_text += new_text
    print(new_text, flush=True, end='')
print(generated_text)

In [ ]:
pip install huggingface_hub

In [ ]:
from huggingface_hub import login
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# Your Hugging Face access token
token_value = 'HF_TOKEN'

# Authenticate with the Hugging Face Hub
login(token=token_value, add_to_git_credential=True)

# Configure quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model and tokenizer with the access token
model = AutoModel.from_pretrained(
    "ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
    use_auth_token=True  # Use the authenticated token
)
tokenizer = AutoTokenizer.from_pretrained("ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1", trust_remote_code=True, use_auth_token=True)

# Load an image and prepare a question
image = Image.open("Path to Your image").convert('RGB')
question = 'Give the modality, organ, analysis, abnormalities (if any), treatment (if abnormalities are present)?'

# Format the input message
msgs = [{'role': 'user', 'content': [image, question]}]

# Generate a response using the model
res = model.chat(
    image=image,
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True,
    temperature=0.95,
    stream=True
)

# Collect and print the generated response
generated_text = ""
for new_text in res:
    generated_text += new_text
    print(new_text, flush=True, end='')
print(generated_text)

In [ ]:
pip install -U bitsandbytes

In [ ]:
import bitsandbytes as bnb
print(bnb.__version__)

In [ ]:
from huggingface_hub import login
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# Your Hugging Face access token
token_value = 'HF_TOKEN'

# Authenticate with the Hugging Face Hub
login(token=token_value, add_to_git_credential=True)

# Configure quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model and tokenizer with the access token
model = AutoModel.from_pretrained(
    "ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
    use_auth_token=True  # Use the authenticated token
)
tokenizer = AutoTokenizer.from_pretrained("ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1", trust_remote_code=True, use_auth_token=True)

# Load an image and prepare a question
image = Image.open("Path to Your image").convert('RGB')
question = 'Give the modality, organ, analysis, abnormalities (if any), treatment (if abnormalities are present)?'

# Format the input message
msgs = [{'role': 'user', 'content': [image, question]}]

# Generate a response using the model
res = model.chat(
    image=image,
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True,
    temperature=0.95,
    stream=True
)

# Collect and print the generated response
generated_text = ""
for new_text in res:
    generated_text += new_text
    print(new_text, flush=True, end='')
print(generated_text)

In [ ]:
!pip install -U bitsandbytes

In [ ]:
from huggingface_hub import login
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# Your Hugging Face access token
token_value = 'HF_TOKEN'

# Authenticate with the Hugging Face Hub
login(token=token_value, add_to_git_credential=True)

# Configure quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the model and tokenizer with the access token
model = AutoModel.from_pretrained(
    "ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
    attn_implementation="flash_attention_2",
    token=token_value  # Use the token argument
)
tokenizer = AutoTokenizer.from_pretrained("ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1", trust_remote_code=True, token=token_value)

# Load an image and prepare a question
image = Image.open("Path to Your image").convert('RGB')
question = 'Give the modality, organ, analysis, abnormalities (if any), treatment (if abnormalities are present)?'

# Format the input message
msgs = [{'role': 'user', 'content': [image, question]}]

# Generate a response using the model
res = model.chat(
    image=image,
    msgs=msgs,
    tokenizer=tokenizer,
    sampling=True,
    temperature=0.95,
    stream=True
)

# Collect and print the generated response
generated_text = ""
for new_text in res:
    generated_text += new_text
    print(new_text, flush=True, end='')
print(generated_text)

In [ ]:
ortError                               Traceback (most recent call last)
<ipython-input-12-365d697a1718> in <cell line: 0>()
     19
     20 # Load the model and tokenizer with the access token
---> 21 model = AutoModel.from_pretrained(
     22     "ContactDoctor/Bio-Medical-MultiModal-Llama-3-8B-V1",
     23     quantization_config=bnb_config,

3 frames
/usr/local/lib/python3.11/dist-packages/transformers/quantizers/quantizer_bnb_4bit.py in validate_environment(self, *args, **kwargs)
     74             )
     75         if not is_bitsandbytes_available():
---> 76             raise ImportError(
     77                 "Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`"
     78             )

ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`

---------------------------------------------------------------------------
NOTE: If your import is failing due to a missing package, you can
manually install dependencies using either !pip or !apt.

To view examples of installing some common dependencies, click the
"Open Examples" button below.